# 🏡 Entrenamiento del Modelo de Predicción de Precios de Casas

## 1. Explicación del Modelo
* **Variable Objetivo ($y$):** `precio` (en USD).
* **Variables de Entrada ($X$):** `ubicacion`, `area`, `habitaciones`, `banos`, `antiguedad`.
* **Algoritmo:** `RandomForestRegressor` (Bosque Aleatorio de Regresión).
* **Métricas de Evaluación:** MAE (Mean Absolute Error) y $R^2$ Score.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np

# Herramientas de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Herramientas de Preprocesamiento y Pipelines
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("✅ Librerías importadas correctamente.")

# --- DETECTOR AUTOMÁTICO DE RUTAS ---
# Esto evita el FileNotFoundError verificando dónde está parado VS Code en este momento
if os.path.exists("data/raw/casas_plusvalia.csv"):
    ruta_raw = "data/raw/casas_plusvalia.csv"
    ruta_proc = "data/processed/casas_limpias.csv"
    ruta_mod = "models/modelo_precios.pkl"
else:
    ruta_raw = "../data/raw/casas_plusvalia.csv"
    ruta_proc = "../data/processed/casas_limpias.csv"
    ruta_mod = "../models/modelo_precios.pkl"

print(f"📂 Ruta detectada para el CSV: {ruta_raw}")

✅ Librerías importadas correctamente.
📂 Ruta detectada para el CSV: ../data/raw/casas_plusvalia.csv


In [2]:
# 1. Cargar el dataset
df = pd.read_csv(ruta_raw)
print(f"📊 Total de registros cargados: {len(df)}")

# 2. Limpieza de datos
df = df.dropna()
df = df[df['precio'] > 10000] # Evitamos precios con errores de tipeo en el portal (< USD 10k)
df = df[df['area'] > 10]      # Evitamos áreas irreales (< 10 m²)
df = df.drop_duplicates()

print(f"✨ Registros limpios y listos para entrenar: {len(df)}")

# 3. Guardar dataset limpio en data/processed/ 
os.makedirs(os.path.dirname(ruta_proc), exist_ok=True)
df.to_csv(ruta_proc, index=False, encoding="utf-8-sig")
print(f"💾 CSV limpio guardado exitosamente en: {ruta_proc}")

# Mostramos las primeras 5 filas para verificar en la demostración
df.head()

📊 Total de registros cargados: 90
✨ Registros limpios y listos para entrenar: 90
💾 CSV limpio guardado exitosamente en: ../data/processed/casas_limpias.csv


,ubicacion,precio,area,habitaciones,banos,antiguedad
0,Quito,133500,142,3,2,5
1,Quito,292500,470,3,2,5
2,Quito,134901,392,1,1,5
3,Quito,206000,150,3,2,5
4,Quito,156999,199,2,1,5


In [3]:
# 1. Separar variables de entrada (X) y variable objetivo (y) [cite: 45]
X = df[['ubicacion', 'area', 'habitaciones', 'banos', 'antiguedad']]
y = df['precio']

# 2. Dividir: 80% para que el modelo estudie, 20% para hacerle el examen final
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Crear el transformador para el texto de 'ubicacion' 
# OneHotEncoder convierte las ciudades en números matemáticos que el algoritmo comprende
preprocesador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['ubicacion'])
    ],
    remainder='passthrough' # Mantiene el resto de columnas numéricas intactas
)

# 4. Empaquetar todo en un Pipeline (Preprocesamiento + Algoritmo) 
modelo_pipeline = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('algoritmo', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 5. ¡ENTRENAR EL MODELO! 🧠
print("🧠 Entrenando la Inteligencia Artificial... Espere unos segundos...")
modelo_pipeline.fit(X_train, y_train)
print("🎉 ¡Modelo entrenado con éxito!")

🧠 Entrenando la Inteligencia Artificial... Espere unos segundos...
🎉 ¡Modelo entrenado con éxito!


In [4]:
# 1. Realizar predicciones con las casas del examen (20% de prueba)
predicciones = modelo_pipeline.predict(X_test)

# 2. Calcular métricas [cite: 45]
mae = mean_absolute_error(y_test, predicciones)
r2 = r2_score(y_test, predicciones)

print("\n--- 📈 RESULTADOS DEL EXAMEN DEL MODELO ---")
print(f"💵 Margen de error promedio (MAE): ± USD {mae:,.2f}")
print(f"🎯 Precisión del modelo (R² Score): {r2:.2f}")

# 3. Exportar y guardar en models/modelo_precios.pkl 
os.makedirs(os.path.dirname(ruta_mod), exist_ok=True)
joblib.dump(modelo_pipeline, ruta_mod)

print(f"\n📦 ¡MODELO Y TRANSFORMADORES EXPORTADOS CON ÉXITO! ")
print(f"📁 Archivo generado: {ruta_mod}")
print("🚀 Ahora estamos listos para conectar esto con FastAPI.")


--- 📈 RESULTADOS DEL EXAMEN DEL MODELO ---
💵 Margen de error promedio (MAE): ± USD 165,643.99
🎯 Precisión del modelo (R² Score): 0.48

📦 ¡MODELO Y TRANSFORMADORES EXPORTADOS CON ÉXITO! 
📁 Archivo generado: ../models/modelo_precios.pkl
🚀 Ahora estamos listos para conectar esto con FastAPI.
